# Notebook 1 (Optimised) — EmbeddingANN with Optuna HPO
### MovieLens Rating Prediction

**Goal**: Train a neural network that learns rich embedding representations for
`genres` and `original_language`, tuned automatically using Optuna.
The saved embeddings will be reused by all classical models in Notebook 2.


## Roadmap

```
STEP 1  — Imports & setup
STEP 2  — Load pre-split data  (train / val / test CSVs)
STEP 3  — Numerical preprocessing  (impute → log1p → scale)
STEP 4  — Label encoding  (genres & language → integer indices)
STEP 5  — PyTorch Dataset & DataLoader
STEP 6  — Model definition  (EmbeddingANN)
STEP 7  — Optuna HPO  (find best hyperparameters automatically)
            ├── 7a  objective function (one trial = one model training run)
            ├── 7b  run the study (50 trials, TPE sampler, MedianPruner)
            └── 7c  inspect best trial
STEP 8  — Retrain final model on best hyperparameters
STEP 9  — Evaluate on train / val / test
STEP 10 — Save embeddings to disk  (.npy weight matrices + .json encoders)
STEP 11 — Feature importance via Integrated Gradients
```

> **Before running**: make sure you have run `pip install optuna captum` once.


---
## Step 1 — Imports & Setup

In [ ]:
# Core
import os, json, time, warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')


In [ ]:
# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data         import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR


In [ ]:
# Scikit-learn helpers
from sklearn.impute        import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics       import mean_squared_error, mean_absolute_error


In [ ]:
# Optuna — automatic hyperparameter search
import optuna
from optuna.pruners  import MedianPruner   # kills bad trials early
from optuna.samplers import TPESampler     # Bayesian (TPE) search strategy
optuna.logging.set_verbosity(optuna.logging.WARNING)  # only show warnings, not every trial


In [ ]:
# Reproducibility + device
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch  : {torch.__version__}")
print(f"Optuna   : {optuna.__version__}")
print(f"Device   : {DEVICE}")


---
## Step 2 — Load Pre-split Data

The CSVs are already merged, cleaned and split — no joining needed.
We also declare which columns play which role here, once, so nothing
is scattered across the notebook.


In [ ]:
DATA_DIR = r"C:\Users\Shivanshu Sirohi\OneDrive\Desktop\Shivanshu\White Paper\Personalization\MovieLens\data"
EMB_DIR  = r"C:\Users\Shivanshu Sirohi\OneDrive\Desktop\Shivanshu\White Paper\Personalization\MovieLens\embeddings"
os.makedirs(EMB_DIR, exist_ok=True)


In [ ]:
df_train = pd.read_csv(DATA_DIR + '\\train.csv')
df_val   = pd.read_csv(DATA_DIR + '\\val.csv')
df_test  = pd.read_csv(DATA_DIR + '\\test.csv')

print(f"Train : {df_train.shape}")
print(f"Val   : {df_val.shape}")
print(f"Test  : {df_test.shape}")


In [ ]:
# ── Declare feature roles (single source of truth for the whole notebook) ──

TARGET   = 'rating'             # what we are predicting

CAT_GENRE = 'genres'            # pipe-separated e.g. "Action|Drama"
CAT_LANG  = 'original_language' # single token e.g. "en"

# These go into the MLP as continuous inputs
NUM_COLS = [
    'imdbRating', 'imdbVotes',
    'tmdbRating', 'tmdbVotes',
    'popularity', 'budget', 'runtime', 'revenue',
    'tag_count',  'avg_relevance', 'max_relevance'
]

# Sanity check — confirm every column exists before going further
missing = [c for c in NUM_COLS + [CAT_GENRE, CAT_LANG, TARGET]
           if c not in df_train.columns]
assert len(missing) == 0, f"Missing columns: {missing}"
print("All columns confirmed.")


---
## Step 3 — Numerical Preprocessing

Three steps applied **in order**, always fit only on train:

| Step | What | Why |
|------|------|-----|
| Fill tag columns with 0 | Missing = user left no tag | 0 is the correct semantic value |
| Median imputation | Other missing numerics | Keeps the row without distorting the distribution |
| log1p | Budget, revenue, votes are heavily right-skewed | Compresses large values so they don't dominate gradients |
| StandardScaler | Normalise to mean=0, std=1 | Stable gradient updates in the ANN |


In [ ]:
# Tag features: missing means the user left no tag → 0 is correct, not "unknown"
for df_ in [df_train, df_val, df_test]:
    df_[['tag_count', 'avg_relevance', 'max_relevance']] =         df_[['tag_count', 'avg_relevance', 'max_relevance']].fillna(0)


In [ ]:
# Median imputation for all other numeric columns
# fit_transform on train, transform on val/test (never leak val/test stats)
imputer = SimpleImputer(strategy='median')
df_train[NUM_COLS] = imputer.fit_transform(df_train[NUM_COLS])
df_val[NUM_COLS]   = imputer.transform(df_val[NUM_COLS])
df_test[NUM_COLS]  = imputer.transform(df_test[NUM_COLS])


In [ ]:
# log1p transform — clip to 0 first so log never sees negatives
for df_ in [df_train, df_val, df_test]:
    df_[NUM_COLS] = np.log1p(df_[NUM_COLS].clip(lower=0))


In [ ]:
# StandardScaler — fit on train only
scaler = StandardScaler()
df_train[NUM_COLS] = scaler.fit_transform(df_train[NUM_COLS])
df_val[NUM_COLS]   = scaler.transform(df_val[NUM_COLS])
df_test[NUM_COLS]  = scaler.transform(df_test[NUM_COLS])

print("Numerical preprocessing done.")
print(f"Mean (train): {df_train[NUM_COLS].mean().round(3).values}")
print(f"Std  (train): {df_train[NUM_COLS].std().round(3).values}")


---
## Step 4 — Label Encoding

`nn.Embedding` is a lookup table — it needs an **integer index**, not a string.
So we map every category to a unique integer.

- Index `0` is always reserved for `<UNK>` (unknown / missing)
- We build the vocabulary **only from train** so val/test can't leak new categories in
- Unseen categories in val/test silently map to index 0

**Language** — one token per row → one integer

**Genres** — pipe-separated multi-token → padded sequence of integers
- `"Action|Drama"` → `[1, 3, -1, -1, ...]`
- `-1` marks padding slots (masked out before mean-pooling)


In [ ]:
# ── Language vocab ──
# fillna first so '<UNK>' doesn't accidentally appear as a real language
lang_series = df_train[CAT_LANG].fillna('<UNK>')
unique_langs = sorted(lang_series.unique().tolist())

# Remove '<UNK>' from the sorted list if present, then prepend it at index 0
if '<UNK>' in unique_langs:
    unique_langs.remove('<UNK>')
lang_vocab = ['<UNK>'] + unique_langs      # index 0 = <UNK>
lang2idx   = {lang: i for i, lang in enumerate(lang_vocab)}

print(f"Language vocab size : {len(lang2idx)}")
print(f"Sample mapping      : { {k: lang2idx[k] for k in list(lang2idx)[:5]} }")


In [ ]:
# ── Genre vocab ──
# Collect every individual genre token from train (split the pipe-separated strings)
all_genre_tokens = set()
for g in df_train[CAT_GENRE].dropna():
    all_genre_tokens.update(g.split('|'))

all_genre_tokens.discard('<UNK>')           # prevent duplicate at index 0
genre_vocab = ['<UNK>'] + sorted(all_genre_tokens)
genre2idx   = {g: i for i, g in enumerate(genre_vocab)}

print(f"Genre vocab size    : {len(genre2idx)}")
print(f"Genre tokens        : {sorted(all_genre_tokens)}")


In [ ]:
# ── Encoding functions ──

MAX_GENRES = 10   # no movie has more than ~8 genres; pad/truncate to this length

def encode_lang(series):
    """Map each language string → integer index. Unknowns → 0."""
    return series.fillna('<UNK>').map(lambda x: lang2idx.get(x, 0)).values

def encode_genres(series):
    """
    Map pipe-separated genre string → padded integer array of shape (MAX_GENRES,).
    Real genre indices are >= 1.  Padding slots are filled with -1.
    """
    result = []
    for g_str in series.fillna('<UNK>'):
        tokens  = g_str.split('|')[:MAX_GENRES]
        indices = [genre2idx.get(t, 0) for t in tokens]
        indices += [-1] * (MAX_GENRES - len(indices))   # right-pad with -1
        result.append(indices)
    return np.array(result, dtype=np.int64)


In [ ]:
# ── Apply encoders to all three splits ──
train_lang   = encode_lang(df_train[CAT_LANG])
val_lang     = encode_lang(df_val[CAT_LANG])
test_lang    = encode_lang(df_test[CAT_LANG])

train_genres = encode_genres(df_train[CAT_GENRE])
val_genres   = encode_genres(df_val[CAT_GENRE])
test_genres  = encode_genres(df_test[CAT_GENRE])

print(f"Genre matrix shape : {train_genres.shape}")
print(f"Sample row         : {train_genres[0]}  ({df_train[CAT_GENRE].iloc[0]})")


In [ ]:
# ── Validation: make sure no index is out of bounds ──
n_genres = len(genre2idx)
n_langs  = len(lang2idx)

assert train_lang.max() < n_langs,     f"Lang index out of range: max={train_lang.max()}, vocab={n_langs}"

valid_mask = train_genres >= 0
assert train_genres[valid_mask].max() < n_genres,     f"Genre index out of range: max={train_genres[valid_mask].max()}, vocab={n_genres}"

print(f"All indices valid.")
print(f"Lang  range : [0, {train_lang.max()}]  vocab size = {n_langs}")
print(f"Genre range : [1, {train_genres[valid_mask].max()}]  vocab size = {n_genres}")


In [ ]:
# ── Save encoders now (NB2 needs these even if you skip rerunning HPO) ──
with open(os.path.join(EMB_DIR, 'genre_encoder.json'), 'w') as f:
    json.dump(genre2idx, f)
with open(os.path.join(EMB_DIR, 'lang_encoder.json'), 'w') as f:
    json.dump(lang2idx, f)
print("Encoders saved.")


---
## Step 5 — PyTorch Dataset & DataLoader

A `Dataset` tells PyTorch how to retrieve **one** sample.
A `DataLoader` wraps it to produce **batches**, handles shuffling, and can
load data in parallel.

Each sample returns a 4-tuple:
```
(numerical_features, genre_indices, lang_index, rating_label)
```


In [ ]:
class MovieDataset(Dataset):
    def __init__(self, num_arr, genre_arr, lang_arr, labels):
        # Convert everything to tensors once, upfront
        self.num    = torch.tensor(num_arr,   dtype=torch.float32)
        self.genres = torch.tensor(genre_arr, dtype=torch.long)
        self.lang   = torch.tensor(lang_arr,  dtype=torch.long)
        self.labels = torch.tensor(labels,    dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.num[idx], self.genres[idx], self.lang[idx], self.labels[idx]


In [ ]:
def make_loader(df, genre_arr, lang_arr, batch_size, shuffle):
    """Convenience wrapper — builds Dataset then wraps in DataLoader."""
    ds = MovieDataset(
        num_arr   = df[NUM_COLS].values.astype(np.float32),
        genre_arr = genre_arr,
        lang_arr  = lang_arr,
        labels    = df[TARGET].values.astype(np.float32)
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)


In [ ]:
# Build loaders with default batch size — will be overridden during HPO
DEFAULT_BATCH = 512
train_loader = make_loader(df_train, train_genres, train_lang, DEFAULT_BATCH, shuffle=True)
val_loader   = make_loader(df_val,   val_genres,   val_lang,   DEFAULT_BATCH, shuffle=False)
test_loader  = make_loader(df_test,  test_genres,  test_lang,  DEFAULT_BATCH, shuffle=False)

# Quick sanity check
num_b, genre_b, lang_b, label_b = next(iter(train_loader))
print(f"Batch shapes — num: {num_b.shape}  genre: {genre_b.shape}  lang: {lang_b.shape}  label: {label_b.shape}")


---
## Step 6 — Model Definition (EmbeddingANN)

The architecture:
```
genres (padded indices) ──► nn.Embedding(n_genres, genre_emb_dim)
                              mean-pool across non-padding tokens ──┐
                                                                     ├──► concat ──► MLP ──► rating
language (single index) ──► nn.Embedding(n_langs, lang_emb_dim) ───┤
                                                                     │
numerical features (11)  ────────────────────────────────────────── ┘
```

Hyperparameters that Optuna will tune:
- `genre_emb_dim`, `lang_emb_dim` — size of each embedding vector
- `hidden1`, `hidden2` — width of the two MLP hidden layers
- `dropout` — dropout rate applied after each hidden layer


In [ ]:
class EmbeddingANN(nn.Module):
    def __init__(self, n_genres, n_langs, n_num,
                 genre_emb_dim, lang_emb_dim,
                 hidden1, hidden2, dropout):
        super().__init__()

        # ── Embedding tables ──
        # padding_idx=0 → the <UNK> token always produces a zero vector
        self.genre_emb = nn.Embedding(n_genres, genre_emb_dim, padding_idx=0)
        self.lang_emb  = nn.Embedding(n_langs,  lang_emb_dim,  padding_idx=0)

        # Store vocab sizes for safe clamping in forward()
        self.n_genres = n_genres
        self.n_langs  = n_langs

        # ── MLP head ──
        mlp_input = genre_emb_dim + lang_emb_dim + n_num
        self.mlp = nn.Sequential(
            nn.Linear(mlp_input, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, 1)           # single output = predicted rating
        )

    def forward(self, num, genre_idx, lang_idx):
        # ── Safety clamp — prevents CUDA out-of-bounds errors ──
        lang_idx_safe  = lang_idx.clamp(0, self.n_langs  - 1)
        genre_idx_safe = genre_idx.clamp(0, self.n_genres - 1)

        # ── Genre: embed each token, mask padding, then mean-pool ──
        mask       = (genre_idx >= 0).float()                     # 1 = real, 0 = padding
        genre_vecs = self.genre_emb(genre_idx_safe)               # (B, MAX_GENRES, emb_dim)
        genre_vecs = genre_vecs * mask.unsqueeze(-1)              # zero out padding positions
        genre_vec  = genre_vecs.sum(1) / mask.sum(1).clamp(min=1).unsqueeze(-1)

        # ── Language: single lookup ──
        lang_vec = self.lang_emb(lang_idx_safe)                   # (B, lang_emb_dim)

        # ── Concatenate and pass through MLP ──
        x = torch.cat([genre_vec, lang_vec, num], dim=1)
        return self.mlp(x).squeeze(1)                             # (B,)


---
## Step 7 — Hyperparameter Optimisation with Optuna

### How Optuna works

```
Study
 └── Trial 1 → suggest hyperparams → train model → return val RMSE
 └── Trial 2 → TPE uses Trial 1 result to suggest better params → ...
 └── Trial N → best params found so far
```

- **TPE sampler** — Bayesian strategy. Learns from previous trials which
  regions of the search space produce low RMSE.
- **MedianPruner** — after each epoch, checks if this trial's val RMSE
  is worse than the median of completed trials at the same epoch.
  If so, it stops the trial early and moves on. Saves a lot of time.

### Search space

| Hyperparameter | Range | Type |
|---|---|---|
| `genre_emb_dim` | 4, 8, 16, 32 | Categorical |
| `lang_emb_dim` | 2, 4, 8 | Categorical |
| `hidden1` | 32 – 256 | Integer |
| `hidden2` | 16 – 128 | Integer |
| `dropout` | 0.0 – 0.5 | Float |
| `lr` | 1e-4 – 1e-2 | Float (log scale) |
| `batch_size` | 256, 512, 1024 | Categorical |


In [ ]:
# ── Helper: one full pass over a DataLoader ──
def run_epoch(model, loader, criterion, optimizer=None):
    """
    Train if optimizer is given, else evaluate.
    Returns (rmse, mae).
    """
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    all_preds, all_labels = [], []

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for num_b, genre_b, lang_b, label_b in loader:
            num_b   = num_b.to(DEVICE)
            genre_b = genre_b.to(DEVICE)
            lang_b  = lang_b.to(DEVICE)
            label_b = label_b.to(DEVICE)

            preds = model(num_b, genre_b, lang_b)
            loss  = criterion(preds, label_b)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(label_b.cpu().numpy())

    preds_arr  = np.array(all_preds)
    labels_arr = np.array(all_labels)
    rmse = np.sqrt(mean_squared_error(labels_arr, preds_arr))
    mae  = mean_absolute_error(labels_arr, preds_arr)
    return rmse, mae


In [ ]:
# ── Optuna objective function ──
# One call to this function = one trial = one full model training run.
# Optuna calls this repeatedly, each time suggesting different hyperparameters.

N_EPOCHS_TRIAL = 10   # shorter training per trial to keep HPO fast
                       # final model will train for longer (Step 8)

def objective(trial):
    # ── 1. Suggest hyperparameters for this trial ──
    genre_emb_dim = trial.suggest_categorical('genre_emb_dim', [4, 8, 16, 32])
    lang_emb_dim  = trial.suggest_categorical('lang_emb_dim',  [2, 4, 8])
    hidden1       = trial.suggest_int('hidden1',   32, 256, step=32)
    hidden2       = trial.suggest_int('hidden2',   16, 128, step=16)
    dropout       = trial.suggest_float('dropout',  0.0, 0.5, step=0.1)
    lr            = trial.suggest_float('lr',       1e-4, 1e-2, log=True)
    batch_size    = trial.suggest_categorical('batch_size', [256, 512, 1024])

    # ── 2. Build loaders with this trial's batch size ──
    tr_loader = make_loader(df_train, train_genres, train_lang, batch_size, shuffle=True)
    v_loader  = make_loader(df_val,   val_genres,   val_lang,   batch_size, shuffle=False)

    # ── 3. Build model with this trial's architecture ──
    model = EmbeddingANN(
        n_genres      = n_genres,
        n_langs       = n_langs,
        n_num         = len(NUM_COLS),
        genre_emb_dim = genre_emb_dim,
        lang_emb_dim  = lang_emb_dim,
        hidden1       = hidden1,
        hidden2       = hidden2,
        dropout       = dropout
    ).to(DEVICE)

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # ── 4. Train for N_EPOCHS_TRIAL epochs ──
    for epoch in range(N_EPOCHS_TRIAL):
        run_epoch(model, tr_loader, criterion, optimizer)     # train
        val_rmse, _ = run_epoch(model, v_loader, criterion)  # evaluate

        # Report to Optuna after each epoch — MedianPruner uses this
        trial.report(val_rmse, epoch)

        # If this trial looks worse than the median so far → stop it early
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return val_rmse   # Optuna minimises this value


In [ ]:
# ── Run the Optuna study ──
# TPESampler = Bayesian search (learns from past trials)
# MedianPruner = stops bad trials early after a warmup of 5 trials

N_TRIALS = 50   # increase for better results; 50 is a good starting point

study = optuna.create_study(
    direction = 'minimize',                 # we want to minimise val RMSE
    sampler   = TPESampler(seed=SEED),
    pruner    = MedianPruner(n_warmup_steps=5)
)

print(f"Starting Optuna search — {N_TRIALS} trials...")
t0 = time.perf_counter()

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

hpo_time = time.perf_counter() - t0
print(f"\nHPO finished in {hpo_time/60:.1f} minutes")
print(f"Trials completed : {len(study.trials)}")
print(f"Trials pruned    : {sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)}")


In [ ]:
# ── Inspect best trial ──
best = study.best_trial
print(f"\nBest val RMSE : {best.value:.4f}")
print(f"\nBest hyperparameters:")
for k, v in best.params.items():
    print(f"  {k:<20} = {v}")


In [ ]:
# ── Visualise: val RMSE across all trials ──
trial_values = [t.value for t in study.trials if t.value is not None]
plt.figure(figsize=(10, 3))
plt.plot(trial_values, alpha=0.5, color='steelblue', label='Val RMSE per trial')
plt.axhline(best.value, color='red', linestyle='--', label=f'Best = {best.value:.4f}')
plt.xlabel('Trial'); plt.ylabel('Val RMSE')
plt.title('Optuna — Val RMSE across trials')
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ── Visualise: hyperparameter importances ──
importances = optuna.importance.get_param_importances(study)
params = list(importances.keys())
values = list(importances.values())

plt.figure(figsize=(8, 3))
plt.barh(params[::-1], values[::-1], color='steelblue')
plt.xlabel('Relative importance')
plt.title('Which hyperparameters matter most?')
plt.tight_layout(); plt.show()


---
## Step 8 — Retrain Final Model on Best Hyperparameters

During HPO each trial only trained for `N_EPOCHS_TRIAL = 10` epochs
(to keep the search fast). Now we retrain from scratch using the best
hyperparameters for the full `N_EPOCHS_FINAL = 30` epochs.

We also add a **cosine decay learning rate scheduler** — this gradually
reduces the learning rate following a cosine curve, helping the model
settle into a better minimum in the final epochs.


In [ ]:
# ── Extract best hyperparameters ──
p = best.params

BEST_BATCH  = p['batch_size']
N_EPOCHS_FINAL = 30

# Rebuild loaders with best batch size
train_loader_final = make_loader(df_train, train_genres, train_lang, BEST_BATCH, shuffle=True)
val_loader_final   = make_loader(df_val,   val_genres,   val_lang,   BEST_BATCH, shuffle=False)
test_loader_final  = make_loader(df_test,  test_genres,  test_lang,  BEST_BATCH, shuffle=False)

print(f"Final training config:")
for k, v in p.items():
    print(f"  {k:<20} = {v}")
print(f"  {'epochs':<20} = {N_EPOCHS_FINAL}")


In [ ]:
# ── Build final model ──
final_model = EmbeddingANN(
    n_genres      = n_genres,
    n_langs       = n_langs,
    n_num         = len(NUM_COLS),
    genre_emb_dim = p['genre_emb_dim'],
    lang_emb_dim  = p['lang_emb_dim'],
    hidden1       = p['hidden1'],
    hidden2       = p['hidden2'],
    dropout       = p['dropout']
).to(DEVICE)

total_params = sum(param.numel() for param in final_model.parameters())
print(f"Model built. Total trainable parameters: {total_params:,}")


In [ ]:
# ── Optimiser + cosine LR scheduler ──
criterion  = nn.MSELoss()
optimizer  = optim.Adam(final_model.parameters(), lr=p['lr'])

# CosineAnnealingLR smoothly reduces lr from p['lr'] down to ~0 over all epochs
# This helps the model converge to a better final minimum
scheduler = CosineAnnealingLR(optimizer, T_max=N_EPOCHS_FINAL, eta_min=1e-6)


In [ ]:
# ── Training loop ──
history = {'train_rmse': [], 'val_rmse': [], 'lr': []}

t0 = time.perf_counter()
print(f"{'Epoch':<8} {'Train RMSE':<14} {'Val RMSE':<14} {'LR'}")
print("-" * 52)

for epoch in range(N_EPOCHS_FINAL):
    tr_rmse, _  = run_epoch(final_model, train_loader_final, criterion, optimizer)
    val_rmse, _ = run_epoch(final_model, val_loader_final,   criterion)
    scheduler.step()    # update learning rate after each epoch

    current_lr = scheduler.get_last_lr()[0]
    history['train_rmse'].append(tr_rmse)
    history['val_rmse'].append(val_rmse)
    history['lr'].append(current_lr)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"{epoch+1:<8} {tr_rmse:<14.4f} {val_rmse:<14.4f} {current_lr:.2e}")

final_train_time = time.perf_counter() - t0
print(f"\nFinal model trained in {final_train_time:.1f}s")


In [ ]:
# ── Learning curves ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_rmse'], label='Train RMSE')
ax1.plot(history['val_rmse'],   label='Val RMSE')
ax1.set_title('RMSE over epochs'); ax1.set_xlabel('Epoch'); ax1.legend()

ax2.plot(history['lr'], color='orange')
ax2.set_title('Learning rate (cosine decay)'); ax2.set_xlabel('Epoch')
ax2.set_ylabel('LR')

plt.tight_layout(); plt.show()


---
## Step 9 — Final Evaluation on All Splits


In [ ]:
ann_tr_rmse,  ann_tr_mae  = run_epoch(final_model, train_loader_final, criterion)
ann_val_rmse, ann_val_mae = run_epoch(final_model, val_loader_final,   criterion)
ann_te_rmse,  ann_te_mae  = run_epoch(final_model, test_loader_final,  criterion)

print("=" * 50)
print("EmbeddingANN (Optimised) — Final Results")
print("=" * 50)
print(f"  Train      RMSE: {ann_tr_rmse:.4f}   MAE: {ann_tr_mae:.4f}")
print(f"  Validation RMSE: {ann_val_rmse:.4f}   MAE: {ann_val_mae:.4f}")
print(f"  Test       RMSE: {ann_te_rmse:.4f}   MAE: {ann_te_mae:.4f}")
print("=" * 50)
print(f"  Train–Test RMSE gap: {ann_te_rmse - ann_tr_rmse:.4f}  (small = no overfitting)")


---
## Step 10 — Save Embeddings to Disk

We extract the weight matrices from the two `nn.Embedding` layers.
Each row = the learned dense vector for one category index.

```
genre_embeddings.npy   shape (n_genres, genre_emb_dim)
lang_embeddings.npy    shape (n_langs,  lang_emb_dim)
genre_encoder.json     {"Action": 1, "Drama": 2, ...}
lang_encoder.json      {"en": 1, "fr": 2, ...}
ann_results.json       RMSE / MAE numbers for NB2's comparison table
best_params.json       Optuna best trial params (useful for reproducibility)
```

Notebook 2 will load these files and swap them in as features
for Linear Regression, Random Forest and XGBoost.


In [ ]:
# ── Extract weight matrices ──
genre_weights = final_model.genre_emb.weight.detach().cpu().numpy()
lang_weights  = final_model.lang_emb.weight.detach().cpu().numpy()

print(f"Genre embedding matrix : {genre_weights.shape}  (n_genres x emb_dim)")
print(f"Lang  embedding matrix : {lang_weights.shape}  (n_langs  x emb_dim)")


In [ ]:
# ── Save weight matrices ──
np.save(os.path.join(EMB_DIR, 'genre_embeddings.npy'), genre_weights)
np.save(os.path.join(EMB_DIR, 'lang_embeddings.npy'),  lang_weights)
print("Embedding weights saved.")


In [ ]:
# ── Save results JSON (NB2 reads this for the final comparison table) ──
ann_results = {
    'model'         : 'EmbeddingANN (Optimised)',
    'train_rmse'    : round(ann_tr_rmse,  4),
    'val_rmse'      : round(ann_val_rmse, 4),
    'test_rmse'     : round(ann_te_rmse,  4),
    'train_mae'     : round(ann_tr_mae,   4),
    'val_mae'       : round(ann_val_mae,  4),
    'test_mae'      : round(ann_te_mae,   4),
    'train_time_s'  : round(final_train_time, 1)
}
with open(os.path.join(EMB_DIR, 'ann_results.json'), 'w') as f:
    json.dump(ann_results, f, indent=2)
print("ann_results.json saved.")


In [ ]:
# ── Save best Optuna params (good for reproducibility / future reference) ──
best_params = {k: v for k, v in best.params.items()}
best_params['val_rmse'] = round(best.value, 4)
with open(os.path.join(EMB_DIR, 'best_params.json'), 'w') as f:
    json.dump(best_params, f, indent=2)
print("best_params.json saved.")


In [ ]:
# ── Confirm all 6 files exist ──
expected = ['genre_embeddings.npy', 'lang_embeddings.npy',
            'genre_encoder.json',    'lang_encoder.json',
            'ann_results.json',      'best_params.json']

print("\nEmbedding folder contents:")
for fname in expected:
    fpath = os.path.join(EMB_DIR, fname)
    status = '✓' if os.path.exists(fpath) else '✗ MISSING'
    print(f"  {status}  {fname}")


---
## Step 11 — Feature Importance via Integrated Gradients

For the ANN we can't use tree-style feature importance (no splits).
Instead we use **Integrated Gradients** from the Captum library.

**Intuition**: starting from an all-zero "no information" baseline,
we interpolate towards the real input in small steps and accumulate
the gradient of the output w.r.t. each input feature along that path.
Features with large accumulated gradients had the most influence on
the prediction.

We run it on 200 validation samples and average to get a global view.


In [ ]:
# !pip install captum
from captum.attr import IntegratedGradients

# ── Wrapper model that accepts one flat tensor ──
# IG requires a single input tensor, so we concatenate everything
# Layout: [num(11) | lang(1) | genres(MAX_GENRES)]
class FlatWrapper(nn.Module):
    def __init__(self, emb_model, n_num, max_g):
        super().__init__()
        self.m     = emb_model
        self.n_num = n_num
        self.max_g = max_g

    def forward(self, x):
        num      = x[:, :self.n_num]
        lang_idx = x[:, self.n_num].long()
        gen_idx  = x[:, self.n_num+1:].long()
        return self.m(num, gen_idx, lang_idx).unsqueeze(1)


In [ ]:
flat_model = FlatWrapper(final_model, len(NUM_COLS), MAX_GENRES).to(DEVICE)
ig = IntegratedGradients(flat_model)

# Build flat validation array
val_flat = np.concatenate([
    df_val[NUM_COLS].values.astype(np.float32),
    val_lang.reshape(-1, 1).astype(np.float32),
    val_genres.astype(np.float32)
], axis=1)

# Compute attributions on 200 samples
N_IG = 200
all_attr = []
final_model.eval()

for i in range(N_IG):
    x_s      = torch.tensor(val_flat[i:i+1], dtype=torch.float32).to(DEVICE)
    baseline = torch.zeros_like(x_s)
    attr, _  = ig.attribute(x_s, baselines=baseline, return_convergence_delta=True)
    all_attr.append(attr.squeeze().detach().cpu().numpy())

global_attr = np.mean(np.abs(np.array(all_attr)), axis=0)


In [ ]:
# ── Plot top 15 features ──
genre_col_names = [f'genre_pos_{i}' for i in range(MAX_GENRES)]
flat_feat_names = NUM_COLS + ['lang_idx'] + genre_col_names

ig_df = (pd.DataFrame({'feature': flat_feat_names, 'importance': global_attr})
           .sort_values('importance', ascending=False)
           .head(15))

plt.figure(figsize=(8, 4))
plt.barh(ig_df['feature'][::-1], ig_df['importance'][::-1], color='steelblue')
plt.xlabel('Mean |Integrated Gradient|')
plt.title('EmbeddingANN — Top 15 Feature Attributions (n=200 val samples)')
plt.tight_layout(); plt.show()

print("\nTop 10 most influential features:")
print(ig_df[['feature','importance']].head(10).to_string(index=False))
